In [0]:
dbutils.widgets.text("bucket", "nyc-taxi-bucket-12313")
dbutils.widgets.text("prefix", "nyctaxi/historical")
bucket = dbutils.widgets.get("bucket")
prefix = dbutils.widgets.get("prefix")
print(bucket)
print(prefix)

In [0]:
## Descubrir los meses disponibles bajo el prefix del backfill
year_dirs = [f for f in dbutils.fs.ls(f"s3://{bucket}/{prefix}") if f.name.startswith("year=")]

months = []
for year_dir in year_dirs:
    year = year_dir.name.replace("year=", "").rstrip("/")
    months_dirs = [f for f in dbutils.fs.ls(year_dir.path) if f.name.startswith("month=")]
    for month_dir in months_dirs:
        month = month_dir.name.replace("month=", "").rstrip("/")
        months.append((year, month, month_dir.path))
        
months.sort()
print(f"Meses encontrados: {len(months)}")

In [0]:
# Leer el schema de cada mes (sin cargar los datos)
schema_by_month = {}
for year, month, path in months:
    cols = set(spark.read.parquet(path).schema.names)
    schema_by_month[f"{year}-{month}"] = cols
    print(f"{year}-{month}_ {len(cols)} columns")

all_columns = sorted(set.union(*schema_by_month.values()))

In [0]:
# Matriz columna x mes - que falta donde
import pandas as pd
matrix = pd.DataFrame(
    {col: [col in schema_by_month[ym] for ym in schema_by_month] for col in all_columns},
    index=list(schema_by_month.keys())
)

display(spark.createDataFrame((matrix.reset_index().rename(columns={"index": "year_month"}))))

In [0]:
## Resumen: Columnas que No estan presentes en todos los meses

inconsistent = [col for col in all_columns if not matrix[col].all()]

if inconsistent:
    print("Columnas inconsistentes entre meses:")
    for col in inconsistent:
        missing_in = [ym for ym in schema_by_month if col not in schema_by_month[ym]]
        print(f" - {col}: falta en {missing_in}")
else:
    print("Todas las columnas estan presentes en todos los meses del rango")
    

In [0]:
## falta 2024-12, lo relleno con 0
KNOWN_GAPS = {
    "cbd_congestion_fee": ["2024-12"]
}

unexpected = []
for col in inconsistent:
    missing_in = [ym for ym in schema_by_month if col not in schema_by_month[ym]]
    if missing_in != KNOWN_GAPS.get(col):
        unexpected.append((col, missing_in))

if unexpected:
    raise ValueError(
        "Columnas inconsistentes sin decision documentada en KNOWN_GAPS, "
        f"requieren evaluacion antes de entrenar: {unexpected}")
elif inconsistent:
    print(f"Todas las inconsistencias estan documentadas en KNOWN_GAPS: {list(KNOWN_GAPS.keys())}")